In [1]:
import helpers.data as data
import helpers.features as features
import joblib
import hmmlearn.hmm as hmm
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from main import gbt_features, hmm_features, train_size, sequence_length

In [2]:
df = data.get_data()
df = features.add_features(df)
df = df.dropna()

Features computed in 3.430s


In [ ]:
for f in hmm_features:
    print(f"{f}  min: {df[f].min():.6g}  max: {df[f].max():.6g}  inf: {np.isinf(df[f]).sum()}")

# Replace inf/-inf with NaN then drop — inf rows would crash the HMM fit
hmm_train = df[hmm_features].iloc[:int(len(df) * train_size)].replace([np.inf, -np.inf], np.nan).dropna()
print(f"\nHMM training rows: {len(hmm_train)}")

hmm_model = hmm.GaussianHMM(n_components=3, covariance_type="diag")
hmm_model.fit(hmm_train)
joblib.dump(hmm_model, "./trained_models/regime_hmm.pkl")

In [3]:
hmm = joblib.load("./trained_models/regime_hmm.pkl")
df["hmm_state"] = hmm.predict(df[hmm_features])

In [4]:
df = features.add_target(df)
train_df = df[gbt_features].iloc[:int(len(df) * train_size)]
targets = df['target'].iloc[:int(len(df) * train_size)].values

def create_sequences(data, target_values):
    feature_data = data[gbt_features].values
    X_sequences = []
    y_targets = []
    
    for i in range(sequence_length, len(data)):
        sequence = feature_data[i - sequence_length:i]
        flattened_sequence = sequence.flatten()
        X_sequences.append(flattened_sequence)
        y_targets.append(target_values[i])
    
    return np.array(X_sequences), np.array(y_targets)

X, Y = create_sequences(train_df, targets)
print(f"Training data shape: X={X.shape}, Y={Y.shape}")
print(f"Target stats: mean={Y.mean():.6f}, std={Y.std():.6f}, min={Y.min():.6f}, max={Y.max():.6f}")

model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    subsample=0.7,
    max_features='sqrt',
    loss='huber',  # Robust to outliers; targets are continuous in [-1, 1]
    validation_fraction=0.2,
    n_iter_no_change=25,
    random_state=39,
    verbose=1
)

model.fit(X, Y)

# Get continuous predictions in [-1, 1]
train_pred = model.predict(X)

# Create test set from remaining data
test_df_full = df.iloc[int(len(df) * train_size):].copy()
test_df_full = test_df_full.dropna(subset=gbt_features + ['target'])
test_df = test_df_full[gbt_features]
test_targets = test_df_full['target'].values

X_test, Y_test = create_sequences(test_df, test_targets)

# Get continuous predictions for test set
test_pred = model.predict(X_test)

# Regression metrics
train_mae = mean_absolute_error(Y, train_pred)
test_mae = mean_absolute_error(Y_test, test_pred)
train_mse = mean_squared_error(Y, train_pred)
test_mse = mean_squared_error(Y_test, test_pred)
train_r2 = r2_score(Y, train_pred)
test_r2 = r2_score(Y_test, test_pred)

print(f"\nTraining Metrics:")
print(f"  MAE:  {train_mae:.6f}")
print(f"  RMSE: {np.sqrt(train_mse):.6f}")
print(f"  R²:   {train_r2:.6f}")

print(f"\nTest Metrics:")
print(f"  MAE:  {test_mae:.6f}")
print(f"  RMSE: {np.sqrt(test_mse):.6f}")
print(f"  R²:   {test_r2:.6f}")

print(f"\nPredictions - Train: [{train_pred.min():.6f}, {train_pred.max():.6f}]")
print(f"Predictions - Test:  [{test_pred.min():.6f}, {test_pred.max():.6f}]")
print(f"Target range - Train: [{Y.min():.6f}, {Y.max():.6f}]")
print(f"Target range - Test:  [{Y_test.min():.6f}, {Y_test.max():.6f}]")

# Feature importance analysis
feature_importance = model.feature_importances_
feature_names_expanded = [f"{feat}_t-{i}" for i in range(sequence_length-1, -1, -1) for feat in gbt_features]
top_indices = np.argsort(feature_importance)[-20:][::-1]
print(f"\nTop 20 Most Important Features:")
for idx in top_indices:
    print(f"  {feature_names_expanded[idx]}: {feature_importance[idx]:.6f}")

# Aggregate importance by original feature
feature_importance_agg = np.zeros(len(gbt_features))
for i, feat in enumerate(gbt_features):
    start_idx = i * sequence_length
    end_idx = (i + 1) * sequence_length
    feature_importance_agg[i] = feature_importance[start_idx:end_idx].sum()

print(f"\nAggregated Feature Importance:")
sorted_feat_idx = np.argsort(feature_importance_agg)[::-1]
for idx in sorted_feat_idx:
    print(f"  {gbt_features[idx]}: {feature_importance_agg[idx]:.6f}")

joblib.dump(model, "./trained_models/gbt.pkl")
print(f"\nModel saved. Best iteration: {model.n_estimators_}")

Training data shape: X=(41032, 176), Y=(41032,)
Target stats: mean=0.017356, std=0.448342, min=-1.000000, max=1.000000
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           0.0995          -0.0131           34.15s
         2           0.0991          -0.0000           33.21s
         3           0.0985          -0.0003           33.13s
         4           0.0984           0.0007           32.88s
         5           0.0982           0.0006           32.79s
         6           0.0975          -0.0001           32.62s
         7           0.0979           0.0014           32.44s
         8           0.0974          -0.0001           32.46s
         9           0.0971           0.0002           32.37s
        10           0.0975           0.0017           32.32s
        20           0.0953           0.0007           31.14s
        30           0.0923          -0.0006           29.74s
        40           0.0906          -0.0020           28.61s
        50  